# Threshold tuning

Load recorded footage, run it through the pipeline offline, and plot each
feature's distribution against the current `config.py` thresholds so you can
retune `min_good`/`max_good`/`tolerance` against real data instead of guessing.

In [ ]:
import sys
sys.path.insert(0, '../src')

import cv2
import matplotlib.pyplot as plt

from config import load_config
from pose_extraction import PoseExtractor
from smoothing import LandmarkEMASmoother
from features import FeatureTracker

cfg = load_config()
extractor = PoseExtractor(cfg)
smoother = LandmarkEMASmoother(cfg)
tracker = FeatureTracker()

In [ ]:
# Point this at a clip in ../data/raw_footage/
video_path = '../data/raw_footage/YOUR_CLIP.mp4'

cap = cv2.VideoCapture(video_path)
elbow_left, hip_rot, guard_left = [], [], []

while True:
    ok, frame = cap.read()
    if not ok:
        break
    ts_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
    pf = extractor.process(frame, ts_ms)
    if not pf.detected:
        continue
    smoothed = smoother.update(pf.landmarks_xy, pf.visibility)
    feats = tracker.update(smoothed, ts_ms)
    elbow_left.append(feats.elbow_extension_deg['left'])
    hip_rot.append(feats.hip_rotation_deg)
    guard_left.append(feats.guard_height_ratio['left'])

cap.release()
extractor.close()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, name, key in zip(
    axes, [elbow_left, hip_rot, guard_left],
    ['Elbow extension (deg)', 'Hip rotation (deg)', 'Guard height ratio'],
    ['elbow_extension_deg', 'hip_rotation_deg', 'guard_height_ratio'],
):
    t = cfg['thresholds'][key]
    ax.hist(data, bins=30)
    ax.axvline(t['min_good'], color='green', linestyle='--', label='min_good')
    ax.axvline(t['max_good'], color='green', linestyle='--', label='max_good')
    ax.set_title(name)
    ax.legend()
plt.tight_layout()